# Enhanced Multi-File DEXA Data Processor

## Advanced batch processing system for comprehensive DEXA analysis

**Multi-File Processing**: Handle 5+ files simultaneously with batch operations
**Flexible File Types**: Support for .txt, .csv, .xlsx, .pdf, images (.tif, .png, .jpeg, .bmp)
**Intelligent Data Cleaning**: Advanced test detection and duplicate removal
**Smart Imputation**: Multiple strategies for missing data with group-based intelligence
**Unified Format**: Standardize timepoints and batch naming across all data sources
**Export Options**: Generate clean CSV/Excel with comprehensive summaries

*Professional toolkit for research-grade DEXA data processing and analysis*

## 1. Import Required Libraries and Setup

Setting up the environment with all necessary libraries for comprehensive DEXA data processing.

In [ ]:
# Essential data processing libraries
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# File processing libraries
from PIL import Image
import zipfile
import tempfile
import uuid

# Optional libraries for advanced file support
try:
    import pdfplumber
    PDF_SUPPORT = True
    print("PDF support enabled")
except ImportError:
    PDF_SUPPORT = False
    print("Warning: PDF support not available - install pdfplumber for PDF processing")

try:
    import xlrd
    import openpyxl
    EXCEL_SUPPORT = True
    print("Advanced Excel support enabled")
except ImportError:
    EXCEL_SUPPORT = False
    print("Warning: Limited Excel support - install xlrd and openpyxl for full .xls/.xlsx processing")

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('default')

# Configuration variables
MAX_FILES = 50  # Maximum files to process in one batch
MAX_FILE_SIZE_MB = 50  # Maximum file size in MB
SUPPORTED_EXTENSIONS = ['.txt', '.csv', '.xlsx', '.xls', '.pdf', '.tif', '.tiff', '.png', '.jpeg', '.jpg', '.img', '.pxl', '.bmp']

print("Enhanced DEXA Processor Initialized")
print(f"Supported file types: {', '.join(SUPPORTED_EXTENSIONS)}")
print(f"Max files per batch: {MAX_FILES}")
print(f"Max file size: {MAX_FILE_SIZE_MB}MB")

## 2. Multi-File Processing Functions

Advanced functions to handle batch processing of multiple DEXA files with validation, progress tracking, and error handling.

In [ ]:
class BatchFileProcessor:
    """
    Advanced batch file processor for DEXA data with intelligent file handling,
    validation, and progress tracking for processing 5+ files simultaneously.
    """
    
    def __init__(self):
        self.processed_files = []
        self.failed_files = []
        self.errors = []
        self.stats = {}
        
    def validate_files(self, file_paths):
        """Validate file paths and extensions before processing"""
        valid_files = []
        invalid_files = []
        
        for file_path in file_paths:
            if not os.path.exists(file_path):
                invalid_files.append(f"File not found: {file_path}")
                continue
                
            # Check file extension
            file_ext = Path(file_path).suffix.lower()
            if file_ext not in SUPPORTED_EXTENSIONS:
                invalid_files.append(f"Unsupported format: {file_path} ({file_ext})")
                continue
                
            # Check file size
            file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
            if file_size_mb > MAX_FILE_SIZE_MB:
                invalid_files.append(f"File too large: {file_path} ({file_size_mb:.1f}MB)")
                continue
                
            valid_files.append(file_path)
            
        if invalid_files:
            print(f"Validation issues found:")
            for issue in invalid_files:
                print(f"  - {issue}")
                
        return valid_files, invalid_files
    
    def load_all_batch_data(self, file_paths, imputation_strategy='smart'):
        """
        Load and process multiple DEXA files with comprehensive error handling
        
        Parameters:
        -----------
        file_paths : list
            List of file paths to process
        imputation_strategy : str
            Strategy for handling missing data: 'leave_nan', 'zero', 'group_median', 'forward_fill', 'smart'
        
        Returns:
        --------
        pandas.DataFrame : Combined and cleaned dataset
        """
        # Validate inputs
        if not file_paths:
            raise ValueError("No file paths provided")
            
        if len(file_paths) > MAX_FILES:
            print(f"Warning: Too many files ({len(file_paths)}). Processing first {MAX_FILES}")
            file_paths = file_paths[:MAX_FILES]
        
        # Validate files
        files_to_process, invalid_files = self.validate_files(file_paths)
        
        if not files_to_process:
            raise ValueError("No valid files to process")
        
        print(f"Starting batch processing of {len(files_to_process)} files...")
        
        all_data = []
        successful_files = 0
        
        for i, file_path in enumerate(files_to_process):
            try:
                self.print_progress(i + 1, len(files_to_process), os.path.basename(file_path))
                
                # Load file based on extension
                file_ext = Path(file_path).suffix.lower()
                
                if file_ext in ['.txt', '.csv']:
                    df = self.parse_txt_csv_file(file_path)
                elif file_ext in ['.xlsx', '.xls']:
                    df = self.parse_excel_file(file_path)
                elif file_ext == '.pdf':
                    df = self.parse_pdf_file(file_path)
                elif file_ext in ['.tif', '.tiff', '.png', '.jpeg', '.jpg', '.bmp']:
                    df = self.scan_dexa_images([file_path])
                else:
                    raise ValueError(f"Unsupported file type: {file_ext}")
                
                if df is not None and not df.empty:
                    # Add metadata
                    df['source_file'] = os.path.basename(file_path)
                    df['file_type'] = file_ext
                    df['processed_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                    
                    all_data.append(df)
                    successful_files += 1
                    self.processed_files.append(file_path)
                else:
                    self.failed_files.append(file_path)
                    self.errors.append(f"No data extracted from {file_path}")
                    
            except Exception as e:
                error_msg = f"Error processing {file_path}: {str(e)}"
                self.errors.append(error_msg)
                self.failed_files.append(file_path)
                print(f"  Error: {error_msg}")
        
        if not all_data:
            raise ValueError("No data could be extracted from any files")
        
        # Combine all data
        combined_df = pd.concat(all_data, ignore_index=True)
        
        # Clean and standardize the combined dataset
        cleaned_df = self.comprehensive_data_cleaning(combined_df)
        
        # Apply imputation strategy
        final_df = self.smart_missing_data_imputation(cleaned_df, strategy=imputation_strategy)
        
        # Store processing statistics
        self.stats = {
            'total_files_attempted': len(files_to_process),
            'successful_files': successful_files,
            'failed_files': len(self.failed_files),
            'total_records': len(final_df),
            'data_quality_score': self.calculate_data_quality_score(final_df)
        }
        
        print(f"Batch processing complete!")
        print(f"Processed: {successful_files}/{len(files_to_process)} files")
        print(f"Total records: {len(all_data)}")
        
        if self.errors:
            print(f"Errors encountered: {len(self.errors)}")
            for error in self.errors[:3]:  # Show first 3 errors
                print(f"  - {error}")
        
        return final_df
    
    def print_progress(self, current, total, filename):
        """Print processing progress"""
        progress = (current / total) * 100
        print(f"Processing {current}/{total} ({progress:.1f}%): {filename}")

print("BatchFileProcessor class created - ready for multi-file operations!")

## 3. Flexible File Type Detection and Parsing

Enhanced file parsing with automatic type detection and format-specific processing for all supported DEXA file types.

In [ ]:
def comprehensive_data_cleaning(df):
    """
    Enhanced data cleaning with comprehensive validation, outlier detection,
    and quality assessment for multi-batch DEXA data processing.
    """
    print("Starting comprehensive data cleaning...")
    
    # Initialize cleaning report
    cleaning_report = {
        'original_records': len(df),
        'test_records_removed': 0,
        'invalid_records_removed': 0,
        'outliers_removed': 0,
        'duplicates_removed': 0,
        'issues_found': []
    }
    
    if df.empty:
        print("Warning: Empty dataset provided")
        return df, cleaning_report
    
    cleaned_df = df.copy()
    initial_count = len(cleaned_df)
    
    # 1. ENHANCED TEST DATA DETECTION
    print("  Detecting and removing test data...")
    
    test_patterns = [
        r'^test', r'^demo', r'^example', r'^sample', r'^dummy',
        r'test$', r'_test', r'-test', r'\.test',
        r'^calib', r'^cal', r'calibration', r'phantom',
        r'^qc', r'quality.*control', r'^qa',
        r'training', r'practice', r'trial',
        r'unknown', r'tbd', r'placeholder'
    ]
    
    test_columns = ['subject_id', 'batch', 'timepoint', 'gender', 'filename']
    
    for col in test_columns:
        if col in cleaned_df.columns:
            for pattern in test_patterns:
                mask = cleaned_df[col].astype(str).str.contains(pattern, case=False, na=False)
                test_rows = mask.sum()
                if test_rows > 0:
                    print(f"    Removing {test_rows} {pattern} rows from {col}")
                    cleaned_df = cleaned_df[~mask]
    
    cleaning_report['test_records_removed'] = initial_count - len(cleaned_df)
    
    # 2. CRITICAL FIELD VALIDATION
    print("  Validating critical fields...")
    
    critical_fields = ['subject_id', 'batch', 'timepoint']
    before_validation = len(cleaned_df)
    
    for field in critical_fields:
        if field in cleaned_df.columns:
            # Remove null/empty values
            null_mask = cleaned_df[field].isnull() | (cleaned_df[field].astype(str).str.strip() == '')
            null_count = null_mask.sum()
            if null_count > 0:
                print(f"    Removing {null_count} records with missing {field}")
                cleaned_df = cleaned_df[~null_mask]
                cleaning_report['issues_found'].append(f"Missing {field}: {null_count} records")
    
    cleaning_report['invalid_records_removed'] = before_validation - len(cleaned_df)
    
    # 3. ENHANCED MEASUREMENT VALIDATION
    print("  Validating DEXA measurements...")
    
    measurement_fields = ['total_weight', 'soft_weight', 'lean_weight', 'fat_weight', 
                         'fat_percent', 'bmc', 'bmd', 'bone_area', 'sample_area']
    
    validation_rules = {
        'total_weight': {'min': 0, 'max': 200, 'unit': 'g'},
        'soft_weight': {'min': 0, 'max': 200, 'unit': 'g'},
        'lean_weight': {'min': 0, 'max': 200, 'unit': 'g'},
        'fat_weight': {'min': 0, 'max': 100, 'unit': 'g'},
        'fat_percent': {'min': 0, 'max': 100, 'unit': '%'},
        'bmc': {'min': 0, 'max': 10, 'unit': 'g'},
        'bmd': {'min': 0, 'max': 200, 'unit': 'mg/cm??'},
        'bone_area': {'min': 0, 'max': 50, 'unit': 'cm??'},
        'sample_area': {'min': 0, 'max': 100, 'unit': 'cm??'}
    }
    
    outliers_before = len(cleaned_df)
    
    for field in measurement_fields:
        if field not in cleaned_df.columns:
            continue
            
        # Convert to numeric first
        cleaned_df[field] = pd.to_numeric(cleaned_df[field], errors='coerce')
        
        rules = validation_rules.get(field, {})
        
        # Apply range validation
        if 'min' in rules and 'max' in rules:
            valid_range = (cleaned_df[field] >= rules['min']) & (cleaned_df[field] <= rules['max'])
            invalid_count = (~valid_range & cleaned_df[field].notna()).sum()
            
            if invalid_count > 0:
                print(f"    Removing {invalid_count} records with invalid {field} values")
                cleaned_df = cleaned_df[valid_range | cleaned_df[field].isna()]
                cleaning_report['issues_found'].append(f"Invalid {field} range: {invalid_count} records")
        
        # Statistical outlier detection (more sophisticated)
        if len(cleaned_df) > 10 and cleaned_df[field].notna().sum() > 5:
            field_data = cleaned_df[field].dropna()
            
            if len(field_data) > 0 and field_data.std() > 0:
                # Use IQR method for outlier detection
                Q1 = field_data.quantile(0.25)
                Q3 = field_data.quantile(0.75)
                IQR = Q3 - Q1
                
                # More lenient outlier bounds
                lower_bound = Q1 - 2.5 * IQR
                upper_bound = Q3 + 2.5 * IQR
                
                outlier_mask = (cleaned_df[field] < lower_bound) | (cleaned_df[field] > upper_bound)
                outlier_count = (outlier_mask & cleaned_df[field].notna()).sum()
                
                if outlier_count > 0 and outlier_count < len(cleaned_df) * 0.1:  # Don't remove more than 10%
                    print(f"    Removing {outlier_count} statistical outliers in {field}")
                    cleaned_df = cleaned_df[~outlier_mask | cleaned_df[field].isna()]
                    cleaning_report['issues_found'].append(f"Statistical outliers in {field}: {outlier_count} records")
    
    cleaning_report['outliers_removed'] = outliers_before - len(cleaned_df)
    
    # 4. ENHANCED BATCH AND TIMEPOINT STANDARDIZATION
    print("  Standardizing batch and timepoint naming...")
    
    # Enhanced batch mapping with more patterns
    if 'batch' in cleaned_df.columns:
        batch_mapping = {
            # Standard patterns
            'batch1': 'Batch_1', 'batch_1': 'Batch_1', 'b1': 'Batch_1', 'batch 1': 'Batch_1',
            'batch2': 'Batch_2', 'batch_2': 'Batch_2', 'b2': 'Batch_2', 'batch 2': 'Batch_2',
            'batch3': 'Batch_3', 'batch_3': 'Batch_3', 'b3': 'Batch_3', 'batch 3': 'Batch_3',
            'batch4': 'Batch_4', 'batch_4': 'Batch_4', 'b4': 'Batch_4', 'batch 4': 'Batch_4',
            'batch5': 'Batch_5', 'batch_5': 'Batch_5', 'b5': 'Batch_5', 'batch 5': 'Batch_5',
            # Extended patterns
            'batch6': 'Batch_6', 'batch_6': 'Batch_6', 'b6': 'Batch_6', 'batch 6': 'Batch_6',
            'batch7': 'Batch_7', 'batch_7': 'Batch_7', 'b7': 'Batch_7', 'batch 7': 'Batch_7',
            'batch8': 'Batch_8', 'batch_8': 'Batch_8', 'b8': 'Batch_8', 'batch 8': 'Batch_8',
            'batch9': 'Batch_9', 'batch_9': 'Batch_9', 'b9': 'Batch_9', 'batch 9': 'Batch_9',
            'batch10': 'Batch_10', 'batch_10': 'Batch_10', 'b10': 'Batch_10', 'batch 10': 'Batch_10',
        }
        
        # Clean and standardize batch names
        cleaned_df['batch'] = cleaned_df['batch'].astype(str).str.strip().str.lower()
        cleaned_df['batch'] = cleaned_df['batch'].map(batch_mapping).fillna(cleaned_df['batch'])
        
        # Count standardized batches
        unique_batches = cleaned_df['batch'].nunique()
        print(f"    Standardized {unique_batches} unique batches")
    
    # Enhanced timepoint standardization
    if 'timepoint' in cleaned_df.columns:
        timepoint_mapping = {
            # Baseline variations
            'week_0': 'Baseline', 'week0': 'Baseline', 'week 0': 'Baseline',
            'pre_scan': 'Baseline', 'prescan': 'Baseline', 'pre scan': 'Baseline',
            'baseline': 'Baseline', 'pre': 'Baseline', 'week-1': 'Baseline',
            # Week patterns
            'week_1': 'Week_1', 'week1': 'Week_1', 'week 1': 'Week_1',
            'week_2': 'Week_2', 'week2': 'Week_2', 'week 2': 'Week_2',
            'week_3': 'Week_3', 'week3': 'Week_3', 'week 3': 'Week_3',
            'week_4': 'Week_4', 'week4': 'Week_4', 'week 4': 'Week_4',
            'week_5': 'Week_5', 'week5': 'Week_5', 'week 5': 'Week_5',
            # Post patterns
            'post_scan': 'Post_Scan', 'postscan': 'Post_Scan', 'post scan': 'Post_Scan',
            'post_treatment': 'Post_Scan', 'post treatment': 'Post_Scan',
            'final': 'Post_Scan', 'end': 'Post_Scan',
            # Other
            'root': 'Unknown', 'unknown': 'Unknown'
        }
        
        # Clean and standardize timepoint names
        cleaned_df['timepoint'] = cleaned_df['timepoint'].astype(str).str.strip().str.lower()
        cleaned_df['timepoint_original'] = cleaned_df['timepoint'].copy()
        cleaned_df['timepoint_standardized'] = cleaned_df['timepoint'].map(timepoint_mapping).fillna(cleaned_df['timepoint'])
        
        unique_timepoints = cleaned_df['timepoint_standardized'].nunique()
        print(f"    Standardized {unique_timepoints} unique timepoints")
    
    # 5. SUBJECT ID CLEANING
    print("  Cleaning subject IDs...")
    
    if 'subject_id' in cleaned_df.columns:
        # Remove problematic subject IDs
        problematic_ids = ['', 'nan', 'none', 'null', 'unknown', 'missing']
        before_id_clean = len(cleaned_df)
        
        cleaned_df['subject_id'] = cleaned_df['subject_id'].astype(str).str.strip()
        id_mask = ~cleaned_df['subject_id'].str.lower().isin(problematic_ids)
        cleaned_df = cleaned_df[id_mask]
        
        id_clean_count = before_id_clean - len(cleaned_df)
        if id_clean_count > 0:
            print(f"    Removed {id_clean_count} records with invalid subject IDs")
    
    # 6. DUPLICATE DETECTION AND REMOVAL
    print("  Detecting and removing duplicates...")
    
    duplicates_before = len(cleaned_df)
    
    # Define columns for duplicate detection (exclude filename for logical duplicates)
    duplicate_columns = ['batch', 'subject_id', 'timepoint', 'gender'] + \
                       [col for col in measurement_fields if col in cleaned_df.columns]
    
    # Remove exact duplicates
    cleaned_df = cleaned_df.drop_duplicates(subset=duplicate_columns, keep='first')
    
    duplicates_removed = duplicates_before - len(cleaned_df)
    cleaning_report['duplicates_removed'] = duplicates_removed
    
    if duplicates_removed > 0:
        print(f"    Removed {duplicates_removed} duplicate records")
    
    # 7. DATA QUALITY ASSESSMENT
    print("  Assessing data quality...")
    
    quality_metrics = {
        'completeness': 0,
        'validity': 0,
        'consistency': 0,
        'accuracy': 0
    }
    
    # Completeness: percentage of non-null values in key fields
    key_fields = ['batch', 'subject_id', 'timepoint', 'gender'] + measurement_fields[:4]
    total_possible_values = len(cleaned_df) * len(key_fields)
    non_null_values = sum(cleaned_df[field].notna().sum() for field in key_fields if field in cleaned_df.columns)
    quality_metrics['completeness'] = (non_null_values / total_possible_values) * 100 if total_possible_values > 0 else 0
    
    # Validity: percentage of values within expected ranges
    valid_count = len(cleaned_df)  # All remaining records passed validation
    total_count = cleaning_report['original_records']
    quality_metrics['validity'] = (valid_count / total_count) * 100 if total_count > 0 else 0
    
    # Consistency: standardization success rate
    if 'batch' in cleaned_df.columns and 'timepoint_standardized' in cleaned_df.columns:
        standardized_batches = cleaned_df['batch'].str.startswith('Batch_').sum()
        standardized_timepoints = cleaned_df['timepoint_standardized'].isin(['Baseline', 'Week_1', 'Week_2', 'Week_3', 'Week_4', 'Week_5', 'Post_Scan']).sum()
        quality_metrics['consistency'] = ((standardized_batches + standardized_timepoints) / (2 * len(cleaned_df))) * 100
    
    # Accuracy: inverse of outliers removed (higher is better)
    outlier_rate = cleaning_report['outliers_removed'] / cleaning_report['original_records'] if cleaning_report['original_records'] > 0 else 0
    quality_metrics['accuracy'] = max(0, (1 - outlier_rate) * 100)
    
    # Overall quality score (weighted average)
    overall_quality = (
        quality_metrics['completeness'] * 0.3 +
        quality_metrics['validity'] * 0.3 +
        quality_metrics['consistency'] * 0.2 +
        quality_metrics['accuracy'] * 0.2
    )
    
    cleaning_report['data_quality_score'] = round(overall_quality, 2)
    cleaning_report['quality_metrics'] = quality_metrics
    cleaning_report['final_records'] = len(cleaned_df)
    
    # FINAL REPORT
    print("\nCOMPREHENSIVE CLEANING REPORT:")
    print(f"  Original records: {cleaning_report['original_records']}")
    print(f"  Test records removed: {cleaning_report['test_records_removed']}")
    print(f"  Invalid records removed: {cleaning_report['invalid_records_removed']}")
    print(f"  Outliers removed: {cleaning_report['outliers_removed']}")
    print(f"  Duplicates removed: {cleaning_report['duplicates_removed']}")
    print(f"  Final records: {cleaning_report['final_records']}")
    print(f"  Data quality score: {cleaning_report['data_quality_score']:.1f}%")
    
    if cleaning_report['issues_found']:
        print(f"  Issues addressed: {len(cleaning_report['issues_found'])}")
        for issue in cleaning_report['issues_found'][:5]:  # Show first 5
            print(f"    ??? {issue}")
        if len(cleaning_report['issues_found']) > 5:
            print(f"    ... and {len(cleaning_report['issues_found']) - 5} more")
    
    return cleaned_df, cleaning_report

print("Comprehensive data cleaning functions created!")
print("Features: Advanced test detection, statistical validation, quality scoring")

## 4. Advanced Data Cleaning and Validation

Comprehensive cleaning functions to remove test data, validate measurements, and standardize formats across all batches.

In [ ]:
def clean_dexa_data_comprehensive(df):
    """
    Comprehensive DEXA data cleaning with enhanced validation and reporting.
    
    Features:
    - Advanced test data detection and removal
    - Intelligent outlier detection with statistical validation
    - Multi-level data validation (syntactic, semantic, statistical)
    - Batch and timepoint standardization
    - Data quality scoring and reporting
    """
    
    print("Starting comprehensive DEXA data cleaning...")
    
    cleaning_report = {
        'original_records': len(df),
        'test_records_removed': 0,
        'invalid_records_removed': 0,
        'outliers_removed': 0,
        'duplicates_removed': 0,
        'final_records': 0,
        'data_quality_score': 0,
        'issues_found': []
    }
    
    cleaned_df = df.copy()
    
    # 1. ENHANCED TEST DATA REMOVAL
    print("  Detecting and removing test/calibration data...")
    
    test_patterns = [
        'test', 'TEST', 'Test', 'calibration', 'CALIBRATION', 'blank', 'BLANK',
        'standard', 'STANDARD', 'control', 'CONTROL', 'phantom', 'PHANTOM',
        'demo', 'DEMO', 'sample', 'SAMPLE', 'qc', 'QC', 'quality', 'QUALITY',
        'check', 'CHECK', 'verify', 'VERIFY', 'calib', 'CALIB'
    ]
    
    initial_count = len(cleaned_df)
    test_columns = ['subject_id', 'filename']
    
    for col in test_columns:
        if col in cleaned_df.columns:
            for pattern in test_patterns:
                mask = cleaned_df[col].astype(str).str.contains(pattern, case=False, na=False)
                test_rows = mask.sum()
                if test_rows > 0:
                    print(f"    Removing {test_rows} {pattern} rows from {col}")
                    cleaned_df = cleaned_df[~mask]
    
    cleaning_report['test_records_removed'] = initial_count - len(cleaned_df)
    
    # 2. CRITICAL FIELD VALIDATION
    print("  Validating critical fields...")
    
    critical_fields = ['subject_id', 'batch', 'timepoint']
    before_validation = len(cleaned_df)
    
    for field in critical_fields:
        if field in cleaned_df.columns:
            # Remove null/empty values
            null_mask = cleaned_df[field].isnull() | (cleaned_df[field].astype(str).str.strip() == '')
            null_count = null_mask.sum()
            if null_count > 0:
                print(f"    Removing {null_count} records with missing {field}")
                cleaned_df = cleaned_df[~null_mask]
                cleaning_report['issues_found'].append(f"Missing {field}: {null_count} records")
    
    cleaning_report['invalid_records_removed'] = before_validation - len(cleaned_df)
    
    # 3. ENHANCED MEASUREMENT VALIDATION
    print("  Validating DEXA measurements...")
    
    measurement_fields = ['total_weight', 'soft_weight', 'lean_weight', 'fat_weight', 
                         'fat_percent', 'bmc', 'bmd', 'bone_area', 'sample_area']
    
    validation_rules = {
        'total_weight': {'min': 0, 'max': 200, 'unit': 'g'},
        'soft_weight': {'min': 0, 'max': 200, 'unit': 'g'},
        'lean_weight': {'min': 0, 'max': 200, 'unit': 'g'},
        'fat_weight': {'min': 0, 'max': 100, 'unit': 'g'},
        'fat_percent': {'min': 0, 'max': 100, 'unit': '%'},
        'bmc': {'min': 0, 'max': 10, 'unit': 'g'},
        'bmd': {'min': 0, 'max': 200, 'unit': 'mg/cm??'},
        'bone_area': {'min': 0, 'max': 50, 'unit': 'cm??'},
        'sample_area': {'min': 0, 'max': 100, 'unit': 'cm??'}
    }
    
    outliers_before = len(cleaned_df)
    
    for field in measurement_fields:
        if field not in cleaned_df.columns:
            continue
            
        # Convert to numeric first
        cleaned_df[field] = pd.to_numeric(cleaned_df[field], errors='coerce')
        
        rules = validation_rules.get(field, {})
        
        # Apply range validation
        if 'min' in rules and 'max' in rules:
            valid_range = (cleaned_df[field] >= rules['min']) & (cleaned_df[field] <= rules['max'])
            invalid_count = (~valid_range & cleaned_df[field].notna()).sum()
            
            if invalid_count > 0:
                print(f"    Removing {invalid_count} records with invalid {field} values")
                cleaned_df = cleaned_df[valid_range | cleaned_df[field].isna()]
                cleaning_report['issues_found'].append(f"Invalid {field} range: {invalid_count} records")
        
        # Statistical outlier detection (more sophisticated)
        if len(cleaned_df) > 10 and cleaned_df[field].notna().sum() > 5:
            field_data = cleaned_df[field].dropna()
            
            if len(field_data) > 0 and field_data.std() > 0:
                # Use IQR method for outlier detection
                Q1 = field_data.quantile(0.25)
                Q3 = field_data.quantile(0.75)
                IQR = Q3 - Q1
                
                # More lenient outlier bounds
                lower_bound = Q1 - 2.5 * IQR
                upper_bound = Q3 + 2.5 * IQR
                
                outlier_mask = (cleaned_df[field] < lower_bound) | (cleaned_df[field] > upper_bound)
                outlier_count = (outlier_mask & cleaned_df[field].notna()).sum()
                
                if outlier_count > 0 and outlier_count < len(cleaned_df) * 0.1:  # Don't remove more than 10%
                    print(f"    Removing {outlier_count} statistical outliers in {field}")
                    cleaned_df = cleaned_df[~outlier_mask | cleaned_df[field].isna()]
                    cleaning_report['issues_found'].append(f"Statistical outliers in {field}: {outlier_count} records")
    
    cleaning_report['outliers_removed'] = outliers_before - len(cleaned_df)
    
    # 4. ENHANCED BATCH AND TIMEPOINT STANDARDIZATION
    print("  Standardizing batch and timepoint naming...")
    
    # Enhanced batch mapping with more patterns
    if 'batch' in cleaned_df.columns:
        batch_mapping = {
            # Standard patterns
            'batch1': 'Batch_1', 'batch_1': 'Batch_1', 'b1': 'Batch_1', 'batch 1': 'Batch_1',
            'batch2': 'Batch_2', 'batch_2': 'Batch_2', 'b2': 'Batch_2', 'batch 2': 'Batch_2',
            'batch3': 'Batch_3', 'batch_3': 'Batch_3', 'b3': 'Batch_3', 'batch 3': 'Batch_3',
            'batch4': 'Batch_4', 'batch_4': 'Batch_4', 'b4': 'Batch_4', 'batch 4': 'Batch_4',
            'batch5': 'Batch_5', 'batch_5': 'Batch_5', 'b5': 'Batch_5', 'batch 5': 'Batch_5',
            # Extended patterns
            'batch6': 'Batch_6', 'batch_6': 'Batch_6', 'b6': 'Batch_6', 'batch 6': 'Batch_6',
            'batch7': 'Batch_7', 'batch_7': 'Batch_7', 'b7': 'Batch_7', 'batch 7': 'Batch_7',
            'batch8': 'Batch_8', 'batch_8': 'Batch_8', 'b8': 'Batch_8', 'batch 8': 'Batch_8',
            'batch9': 'Batch_9', 'batch_9': 'Batch_9', 'b9': 'Batch_9', 'batch 9': 'Batch_9',
            'batch10': 'Batch_10', 'batch_10': 'Batch_10', 'b10': 'Batch_10', 'batch 10': 'Batch_10',
        }
        
        # Clean and standardize batch names
        cleaned_df['batch'] = cleaned_df['batch'].astype(str).str.strip().str.lower()
        cleaned_df['batch'] = cleaned_df['batch'].map(batch_mapping).fillna(cleaned_df['batch'])
        
        # Count standardized batches
        unique_batches = cleaned_df['batch'].nunique()
        print(f"    Standardized {unique_batches} unique batches")
    
    # Enhanced timepoint standardization
    if 'timepoint' in cleaned_df.columns:
        timepoint_mapping = {
            # Baseline variations
            'week_0': 'Baseline', 'week0': 'Baseline', 'week 0': 'Baseline',
            'pre_scan': 'Baseline', 'prescan': 'Baseline', 'pre scan': 'Baseline',
            'baseline': 'Baseline', 'pre': 'Baseline', 'week-1': 'Baseline',
            # Week patterns
            'week_1': 'Week_1', 'week1': 'Week_1', 'week 1': 'Week_1',
            'week_2': 'Week_2', 'week2': 'Week_2', 'week 2': 'Week_2',
            'week_3': 'Week_3', 'week3': 'Week_3', 'week 3': 'Week_3',
            'week_4': 'Week_4', 'week4': 'Week_4', 'week 4': 'Week_4',
            'week_5': 'Week_5', 'week5': 'Week_5', 'week 5': 'Week_5',
            # Post patterns
            'post_scan': 'Post_Scan', 'postscan': 'Post_Scan', 'post scan': 'Post_Scan',
            'post_treatment': 'Post_Scan', 'post treatment': 'Post_Scan',
            'final': 'Post_Scan', 'end': 'Post_Scan',
            # Other
            'root': 'Unknown', 'unknown': 'Unknown'
        }
        
        # Clean and standardize timepoint names
        cleaned_df['timepoint'] = cleaned_df['timepoint'].astype(str).str.strip().str.lower()
        cleaned_df['timepoint_original'] = cleaned_df['timepoint'].copy()
        cleaned_df['timepoint_standardized'] = cleaned_df['timepoint'].map(timepoint_mapping).fillna(cleaned_df['timepoint'])
        
        unique_timepoints = cleaned_df['timepoint_standardized'].nunique()
        print(f"    Standardized {unique_timepoints} unique timepoints")
    
    # 5. SUBJECT ID CLEANING
    print("  Cleaning subject IDs...")
    
    if 'subject_id' in cleaned_df.columns:
        # Remove problematic subject IDs
        problematic_ids = ['', 'nan', 'none', 'null', 'unknown', 'missing']
        before_id_clean = len(cleaned_df)
        
        cleaned_df['subject_id'] = cleaned_df['subject_id'].astype(str).str.strip()
        id_mask = ~cleaned_df['subject_id'].str.lower().isin(problematic_ids)
        cleaned_df = cleaned_df[id_mask]
        
        id_clean_count = before_id_clean - len(cleaned_df)
        if id_clean_count > 0:
            print(f"    Removed {id_clean_count} records with invalid subject IDs")
    
    # 6. DUPLICATE DETECTION AND REMOVAL
    print("  Detecting and removing duplicates...")
    
    duplicates_before = len(cleaned_df)
    
    # Define columns for duplicate detection (exclude filename for logical duplicates)
    duplicate_columns = ['batch', 'subject_id', 'timepoint', 'gender'] + \
                       [col for col in measurement_fields if col in cleaned_df.columns]
    
    # Remove exact duplicates
    cleaned_df = cleaned_df.drop_duplicates(subset=duplicate_columns, keep='first')
    
    duplicates_removed = duplicates_before - len(cleaned_df)
    cleaning_report['duplicates_removed'] = duplicates_removed
    
    if duplicates_removed > 0:
        print(f"    Removed {duplicates_removed} duplicate records")
    
    # 7. DATA QUALITY ASSESSMENT
    print("  Assessing data quality...")
    
    quality_metrics = {
        'completeness': 0,
        'validity': 0,
        'consistency': 0,
        'accuracy': 0
    }
    
    # Completeness: percentage of non-null values in key fields
    key_fields = ['batch', 'subject_id', 'timepoint', 'gender'] + measurement_fields[:4]
    total_possible_values = len(cleaned_df) * len(key_fields)
    non_null_values = sum(cleaned_df[field].notna().sum() for field in key_fields if field in cleaned_df.columns)
    quality_metrics['completeness'] = (non_null_values / total_possible_values) * 100 if total_possible_values > 0 else 0
    
    # Validity: percentage of values within expected ranges
    valid_count = len(cleaned_df)  # All remaining records passed validation
    total_count = cleaning_report['original_records']
    quality_metrics['validity'] = (valid_count / total_count) * 100 if total_count > 0 else 0
    
    # Consistency: standardization success rate
    if 'batch' in cleaned_df.columns and 'timepoint_standardized' in cleaned_df.columns:
        standardized_batches = cleaned_df['batch'].str.startswith('Batch_').sum()
        standardized_timepoints = cleaned_df['timepoint_standardized'].isin(['Baseline', 'Week_1', 'Week_2', 'Week_3', 'Week_4', 'Week_5', 'Post_Scan']).sum()
        quality_metrics['consistency'] = ((standardized_batches + standardized_timepoints) / (2 * len(cleaned_df))) * 100
    
    # Accuracy: inverse of outliers removed (higher is better)
    outlier_rate = cleaning_report['outliers_removed'] / cleaning_report['original_records'] if cleaning_report['original_records'] > 0 else 0
    quality_metrics['accuracy'] = max(0, (1 - outlier_rate) * 100)
    
    # Overall quality score (weighted average)
    overall_quality = (
        quality_metrics['completeness'] * 0.3 +
        quality_metrics['validity'] * 0.3 +
        quality_metrics['consistency'] * 0.2 +
        quality_metrics['accuracy'] * 0.2
    )
    
    cleaning_report['data_quality_score'] = round(overall_quality, 2)
    cleaning_report['quality_metrics'] = quality_metrics
    cleaning_report['final_records'] = len(cleaned_df)
    
    # FINAL REPORT
    print("\nCOMPREHENSIVE CLEANING REPORT:")
    print(f"  Original records: {cleaning_report['original_records']}")
    print(f"  Test records removed: {cleaning_report['test_records_removed']}")
    print(f"  Invalid records removed: {cleaning_report['invalid_records_removed']}")
    print(f"  Outliers removed: {cleaning_report['outliers_removed']}")
    print(f"  Duplicates removed: {cleaning_report['duplicates_removed']}")
    print(f"  Final records: {cleaning_report['final_records']}")
    print(f"  Data quality score: {cleaning_report['data_quality_score']:.1f}%")
    
    if cleaning_report['issues_found']:
        print(f"  Issues addressed: {len(cleaning_report['issues_found'])}")
        for issue in cleaning_report['issues_found'][:5]:  # Show first 5
            print(f"    ??? {issue}")
        if len(cleaning_report['issues_found']) > 5:
            print(f"    ... and {len(cleaning_report['issues_found']) - 5} more")
    
    return cleaned_df, cleaning_report

print("Comprehensive data cleaning functions created!")
print("Features: Advanced test detection, statistical validation, quality scoring")

## 5. Data Correlator For Bone Density


Loads and consolidates bone-density exports (HEMAVET/CSV/XLSX/TXT), extracting measurement IDs from filenames when needed and producing a combined bone-data table ready for mapping to subject/mouse identifiers.

In [ ]:
#!/usr/bin/env python3
"""
Boilerplate Python code for data parsing.
Supports CSV, JSON, and plain text formats.
"""

import argparse
import csv
import json
import sys
import os
import re
from typing import List, Dict, Any, Tuple, Optional, Union

import pandas as pd


def parse_csv(filepath: str) -> List[Dict[str, Any]]:
    """Parse a CSV file into a list of dictionaries."""
    try:
        with open(filepath, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            return list(reader)
    except Exception as e:
        sys.exit(f"Error parsing CSV: {e}")


def parse_json(filepath: str) -> Any:
    """Parse a JSON file into Python objects."""
    try:
        with open(filepath, encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        sys.exit(f"Error parsing JSON: {e}")


def parse_text(filepath: str) -> List[str]:
    """Parse a plain text file line by line."""
    try:
        with open(filepath, encoding="utf-8") as f:
            return [line.strip() for line in f if line.strip()]
    except Exception as e:
        sys.exit(f"Error parsing text file: {e}")


def load_bone_density_data(
    hemavet_dir: str,
    mapping: Optional[Union[str, pd.DataFrame]] = None,
    on: str = "measurement_id",
    filename_pattern: Optional[str] = None,
    extensions: Optional[List[str]] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Load bone density files from a directory (HEMAVET export) and link them to mice using a mapping.

    Parameters
    - hemavet_dir: path to directory containing HEMAVET export files (CSV/Excel/text). Can contain many files.
    - mapping: either a path to a CSV mapping file or a pandas.DataFrame. The mapping must contain the column named by `on` and the `mouse_id` column (or similar).
    - on: column name to join on (default 'measurement_id'). The function will attempt to extract this value from filenames if not present in file contents.
    - filename_pattern: optional regex pattern to extract the measurement id from the filename. Use a capturing group, e.g. r"(M\d{3})". If None, the filename (stem) is used.
    - extensions: list of file extensions to consider. Defaults to ['.csv', '.xlsx', '.xls', '.txt'].

    Returns:
    - merged_df: DataFrame with bone density rows and mapping columns attached.
    - unmatched_df: subset of merged_df where the join failed (no mouse linked).

    Notes: this function uses a permissive read strategy: CSV and Excel are attempted first, text files are read as CSV with automatic parsing.
    """
    if extensions is None:
        extensions = [".csv", ".xlsx", ".xls", ".txt"]

    hemavet_dir = os.path.abspath(hemavet_dir)
    if not os.path.isdir(hemavet_dir):
        raise FileNotFoundError(f"HEMAVET directory not found: {hemavet_dir}")

    records: List[pd.DataFrame] = []

    def extract_id_from_filename(fname: str) -> str:
        stem = os.path.splitext(os.path.basename(fname))[0]
        if filename_pattern:
            m = re.search(filename_pattern, stem)
            return m.group(1) if m else stem
        return stem

    for root, _, files in os.walk(hemavet_dir):
        for fname in files:
            lower = fname.lower()
            if not any(lower.endswith(ext) for ext in extensions):
                continue
            path = os.path.join(root, fname)
            try:
                if lower.endswith(('.xlsx', '.xls')):
                    df = pd.read_excel(path)
                else:
                    # try CSV; allow different separators
                    try:
                        df = pd.read_csv(path, engine='python')
                    except Exception:
                        # fallback to reading as whitespace-delimited text
                        df = pd.read_csv(path, sep='\s+', engine='python')
                # annotate source filename and attempt to set measurement id column if missing
                df = df.copy()
                df['__source_file'] = path
                if on not in df.columns:
                    df[on] = extract_id_from_filename(fname)
                records.append(df)
            except Exception as e:
                print(f"Warning: failed to read {path}: {e}")

    if not records:
        print(f"No HEMAVET files found under {hemavet_dir} with extensions {extensions}")
        return pd.DataFrame(), pd.DataFrame()

    combined = pd.concat(records, ignore_index=True, sort=False)

    # Load mapping
    if mapping is None:
        mapping_df = pd.DataFrame()
    elif isinstance(mapping, str):
        mapping_df = pd.read_csv(mapping)
    elif isinstance(mapping, pd.DataFrame):
        mapping_df = mapping.copy()
    else:
        raise TypeError("mapping must be a path or a pandas.DataFrame or None")

    if mapping_df.empty:
        print("Warning: mapping is empty or not provided ??? returning combined raw data")
        return combined, combined

    # Perform left merge: keep all bone density rows and attach mouse metadata
    merged = combined.merge(mapping_df, how='left', on=on, indicator=True)
    unmatched = merged[merged['_merge'] == 'left_only'].copy()
    # drop merge indicator
    merged.drop(columns=['_merge'], inplace=True)
    unmatched.drop(columns=['_merge'], inplace=True, errors='ignore')

    return merged, unmatched


def main():
    parser = argparse.ArgumentParser(description="Generic data parser.")
    parser.add_argument("filepath", help="Path to the data file")
    parser.add_argument(
        "--format",
        choices=["csv", "json", "text"],
        required=True,
        help="File format to parse",
    )
    args = parser.parse_args()

    if args.format == "csv":
        data = parse_csv(args.filepath)
    elif args.format == "json":
        data = parse_json(args.filepath)
    elif args.format == "text":
        data = parse_text(args.filepath)
    else:
        sys.exit("Unsupported format.")

    print("Parsed data:")
    print(data)


if __name__ == "__main__":
    main()


## 6. Correlator With Cleaned Data

Merges cleaned DEXA outputs with aggregated bone-density metrics to compute subject-level baseline???Week_4 absolute and percent changes (BMD/BMC/etc.), attach treatment groups (e.g., FL5 vs IgG), and flag data-quality issues.

In [ ]:
# Correlator: merge cleaned DEXA outputs with bone-density (HEMAVET) data
# Enhanced: subject normalization, treatment-group analysis (FL5 vs IgG),
# longitudinal Week0->Week4 deltas, BMD change (abs & %), and data-quality validation

def correlate_with_cleaned_data(
    cleaned_df=None,
    cleaned_dir=None,
    bone_df=None,
    join_on=None,
    timepoint_col=None,
    baseline_labels=None,
    followup_label="Week_4",
    metrics=None,
    aggfunc="mean",
    treatment_cols=None,
    treatment_map=None,
    perform_stats=True,
):
    """
    Merge cleaned DEXA records with bone-density measurements and compute longitudinal changes
    with data-quality checks and treatment-group comparisons.

    Key capabilities implemented:
    - Subject ID matching (tolerant normalization)
    - Treatment group attachment and group-level summaries (supports FL5 vs IgG)
    - Longitudinal tracking: baseline -> followup (Week_4 by default)
    - BMD (and other metrics) absolute and percent change calculations
    - Data quality validation: match rates, missing baseline/followup flags, per-subject quality score

    Parameters
    - cleaned_df: pandas.DataFrame (preferred). If None, cleaned_dir will be scanned for CSVs and concatenated.
    - cleaned_dir: directory path containing cleaned CSV outputs (used if cleaned_df is None).
    - bone_df: pandas.DataFrame of bone measurements (preferably already merged with mapping to subject ids).
    - join_on: column name used to match subjects across datasets (e.g. 'subject_id' or 'mouse_id'). If None, autodetect.
    - timepoint_col: name of timepoint column (if None autodetects 'timepoint_standardized' or 'timepoint').
    - baseline_labels: list of labels to treat as baseline (e.g. ['Baseline','Week_0','week0']). If None defaults to ['Baseline','Week_0','Week0','week_0'].
    - followup_label: the followup timepoint to compare against baseline (default 'Week_4').
    - metrics: list of numeric metrics to compute changes for. Defaults to ['bmd','bmc','bone_area','sample_area'].
    - aggfunc: aggregation for multiple measurements per subject/timepoint (default 'mean').
    - treatment_cols: list of column names to look for treatment/group in cleaned_df (defaults common names).
    - treatment_map: optional dict mapping raw treatment labels to canonical ones (e.g. {'FL5':'FL5','IgG':'IgG'}).
    - perform_stats: if True, try to run basic t-test between FL5 and IgG (scipy required) and include p-values in output.

    Returns
    - merged_df: cleaned records left-joined with aggregated bone metrics per subject/timepoint
    - summary_df: per-subject pivot with baseline and followup values, delta columns, percent changes, treatment, and quality flags
    - report: dict with high-level merge stats and data-quality numbers
    """
    import glob
    import numpy as np
    from pathlib import Path

    # defaults
    if metrics is None:
        metrics = ["bmd", "bmc", "bone_area", "sample_area"]
    if baseline_labels is None:
        baseline_labels = ["Baseline", "Week_0", "Week0", "week_0", "week0"]
    if treatment_cols is None:
        treatment_cols = ["treatment", "treatment_group", "group", "treatment_group_name"]
    if treatment_map is None:
        treatment_map = {"FL5": "FL5", "IgG": "IgG"}  # canonical map; extend as needed

    # 1) Load cleaned data if needed
    if cleaned_df is None:
        if not cleaned_dir:
            raise ValueError("Either cleaned_df or cleaned_dir must be provided")
        cleaned_dir = os.path.abspath(cleaned_dir)
        csv_paths = sorted(glob.glob(os.path.join(cleaned_dir, "*.csv")))
        if not csv_paths:
            raise FileNotFoundError(f"No CSV files found in cleaned_dir: {cleaned_dir}")
        parts = []
        for p in csv_paths:
            try:
                parts.append(pd.read_csv(p))
            except Exception:
                parts.append(pd.read_csv(p, engine='python'))
        cleaned_df = pd.concat(parts, ignore_index=True, sort=False)

    # 2) Autodetect join_on and timepoint_col if not provided
    cols_lower = {c.lower(): c for c in cleaned_df.columns}
    if join_on is None:
        for candidate in ("subject_id", "mouse_id", "subject", "id", "measurement_id"):
            if candidate in cols_lower:
                join_on = cols_lower[candidate]
                break
        if join_on is None:
            raise KeyError("Could not autodetect subject id column; please pass join_on explicitly")
    if timepoint_col is None:
        for candidate in ("timepoint_standardized", "timepoint", "timepoint_standardised"):
            if candidate in cols_lower:
                timepoint_col = cols_lower[candidate]
                break
        if timepoint_col is None:
            raise KeyError("Could not detect a timepoint column; please pass timepoint_col explicitly")

    # 3) Normalize subject ids in both frames for tolerant matching
    def _normalize_ids(s):
        return s.astype(str).str.strip().str.upper()

    cleaned_df = cleaned_df.copy()
    cleaned_df['_norm_subject'] = _normalize_ids(cleaned_df[join_on])
    cleaned_df['_norm_timepoint'] = cleaned_df[timepoint_col].astype(str).str.strip()

    # 4) Prepare bone_df
    if bone_df is None or bone_df.empty:
        print("Warning: bone_df not provided or empty. Returning cleaned_df only with report indicating missing bone data.")
        report = {
            'cleaned_rows': len(cleaned_df),
            'bone_rows': 0,
            'merged_rows': 0,
            'unmatched_bone': 0,
            'unmatched_cleaned': len(cleaned_df),
        }
        return cleaned_df, pd.DataFrame(), report

    bone = bone_df.copy()
    # try to find subject column in bone and normalize
    bone_cols_lower = {c.lower(): c for c in bone.columns}
    if join_on.lower() not in bone_cols_lower:
        # try subject_id or mouse_id
        for candidate in ("subject_id", "mouse_id", "measurement_id", "id"):
            if candidate in bone_cols_lower:
                bone[join_on] = bone[bone_cols_lower[candidate]]
                break
    bone['_norm_subject'] = _normalize_ids(bone[join_on])

    # ensure timepoint col in bone
    if timepoint_col not in bone.columns:
        for candidate in ('timepoint_standardized', 'timepoint'):
            if candidate in bone_cols_lower:
                bone[timepoint_col] = bone[bone_cols_lower[candidate]]
                break
    bone['_norm_timepoint'] = bone[timepoint_col].astype(str).str.strip()

    # numeric conversion for metrics
    for m in metrics:
        if m in bone.columns:
            bone[m] = pd.to_numeric(bone[m], errors='coerce')

    # 5) Aggregate bone metrics per normalized subject + normalized timepoint
    group_cols = ['_norm_subject', '_norm_timepoint']
    agg_dict = {m: aggfunc for m in metrics if m in bone.columns}
    if not agg_dict:
        print("No requested metrics found in bone_df; returning cleaned_df only")
        report = {'cleaned_rows': len(cleaned_df), 'bone_rows': len(bone), 'merged_rows': 0}
        return cleaned_df.copy(), pd.DataFrame(), report

    bone_agg = bone.groupby(group_cols, as_index=False).agg(agg_dict)

    # 6) Merge cleaned records with bone metrics
    merged_df = cleaned_df.merge(
        bone_agg,
        how='left',
        left_on=['_norm_subject', '_norm_timepoint'],
        right_on=['_norm_subject', '_norm_timepoint'],
        suffixes=('', '_bone')
    )

    # 7) Data-quality reporting: counts and unmatched
    total_cleaned = len(cleaned_df)
    total_bone = len(bone)
    merged_count = merged_df.dropna(subset=list(agg_dict.keys())).drop_duplicates(subset=['_norm_subject', '_norm_timepoint']).shape[0]
    # unmatched bone rows (those without mapped subject after normalization)
    unmatched_bone = bone[bone['_norm_subject'].isna() | (bone['_norm_subject'] == '')]
    # unmatched cleaned subjects (no bone data for subject/timepoint)
    unmatched_cleaned = merged_df[merged_df[list(agg_dict.keys())].isna().all(axis=1)]

    report = {
        'cleaned_rows': total_cleaned,
        'bone_rows': total_bone,
        'merged_subject_timepoints': int(merged_count),
        'unmatched_bone_rows': int(len(unmatched_bone)),
        'unmatched_cleaned_rows': int(len(unmatched_cleaned)),
    }

    # 8) Build per-subject pivot (baseline & followup) using normalized subject
    pivots = []
    for m in agg_dict.keys():
        tmp = bone_agg[['_norm_subject', '_norm_timepoint', m]].dropna()
        if tmp.empty:
            continue
        piv = tmp.pivot_table(index='_norm_subject', columns='_norm_timepoint', values=m, aggfunc='first')
        # rename columns to friendly names e.g. bmd__Baseline
        piv.columns = [f"{m}__{col}" for col in piv.columns.astype(str)]
        pivits = piv
        pivots.append(piv)

    if pivots:
        piv_all = pd.concat(pivots, axis=1, sort=False).reset_index()
    else:
        piv_all = pd.DataFrame({'_norm_subject': pd.Series(dtype=cleaned_df['_norm_subject'].dtype)})

    summary = piv_all.copy()

    # 9) Attach treatment group (canonicalized) if available
    treatment_col_found = None
    for tc in treatment_cols:
        if tc in cleaned_df.columns:
            treatment_col_found = tc
            break
    if treatment_col_found:
        # per-subject treatment: choose first non-null value seen in cleaned_df
        treatments = (
            cleaned_df[['_norm_subject', treatment_col_found]]
            .dropna()
            .drop_duplicates(subset=['_norm_subject'])
            .set_index('_norm_subject')
        )
        treatments = treatments.rename(columns={treatment_col_found: 'treatment_raw'})
        # canonicalize using treatment_map if provided
        treatments['treatment'] = treatments['treatment_raw'].map(lambda x: treatment_map.get(str(x).strip(), str(x).strip()))
        summary = summary.set_index('_norm_subject').join(treatments[['treatment']]).reset_index()
    else:
        summary = summary

    # 10) Compute baseline->followup deltas for each metric
    # determine baseline column name present
    baseline_col_name = None
    for lbl in baseline_labels:
        # find any column that endswith __<lbl>
        candidate_cols = [c for c in summary.columns if isinstance(c, str) and c.endswith(f"__{lbl}")]
        if candidate_cols:
            baseline_col_name = lbl
            break
    if baseline_col_name is None:
        # fallback to 'Baseline'
        baseline_col_name = baseline_labels[0]

    # ensure followup column exists as well
    # compute deltas
    for m in agg_dict.keys():
        base_col = f"{m}__{baseline_col_name}"
        follow_col = f"{m}__{followup_label}"
        if base_col in summary.columns and follow_col in summary.columns:
            abs_col = f"{m}_delta_{followup_label}"
            pct_col = f"{m}_pct_change_{followup_label}"
            summary[abs_col] = summary[follow_col] - summary[base_col]
            summary[pct_col] = np.where(
                pd.notna(summary[base_col]) & (summary[base_col] != 0),
                (summary[abs_col] / summary[base_col].astype(float)) * 100.0,
                np.nan,
            )

    # 11) Per-subject data quality flags
    # flag if missing baseline or missing followup for any metric
    summary['missing_baseline'] = False
    summary['missing_followup'] = False
    for m in agg_dict.keys():
        bcol = f"{m}__{baseline_col_name}"
        fcol = f"{m}__{followup_label}"
        if bcol in summary.columns:
            summary['missing_baseline'] = summary['missing_baseline'] | summary[bcol].isna()
        else:
            summary['missing_baseline'] = True
        if fcol in summary.columns:
            summary['missing_followup'] = summary['missing_followup'] | summary[fcol].isna()
        else:
            summary['missing_followup'] = True

    # simple quality score: percent of requested metrics present at baseline and followup (0-100)
    def _metric_presence_score(row):
        total = 0
        present = 0
        for m in agg_dict.keys():
            total += 2  # baseline + followup
            bcol = f"{m}__{baseline_col_name}"
            fcol = f"{m}__{followup_label}"
            if bcol in row and pd.notna(row[bcol]):
                present += 1
            if fcol in row and pd.notna(row[fcol]):
                present += 1
        return (present / total) * 100 if total > 0 else np.nan

    if not summary.empty:
        summary['data_quality_pct'] = summary.apply(_metric_presence_score, axis=1)
    else:
        summary['data_quality_pct'] = pd.Series(dtype=float)

    # 12) Treatment group analysis: compute group means/sem and optionally t-test between FL5 and IgG
    group_stats = None
    ttest_results = None
    if 'treatment' in summary.columns:
        group_stats = summary.groupby('treatment').agg({
            col: ['count', 'mean', 'std'] for col in summary.columns if col.endswith(f"_delta_{followup_label}") or col.endswith(f"_pct_change_{followup_label}")
        })
        # simplify column names
        group_stats.columns = ['_'.join(c).strip() for c in group_stats.columns.values]
        group_stats = group_stats.reset_index()

        # attempt t-test FL5 vs IgG on each delta metric
        if perform_stats:
            try:
                from scipy import stats
                ttest_results = {}
                for col in summary.columns:
                    if col.endswith(f"_delta_{followup_label}") or col.endswith(f"_pct_change_{followup_label}"):
                        a = summary[summary['treatment'] == 'FL5'][col].dropna()
                        b = summary[summary['treatment'] == 'IgG'][col].dropna()
                        if len(a) >= 2 and len(b) >= 2:
                            tstat, pval = stats.ttest_ind(a, b, equal_var=False)
                            ttest_results[col] = {'tstat': float(tstat), 'pvalue': float(pval), 'n_FL5': int(len(a)), 'n_IgG': int(len(b))}
                        else:
                            ttest_results[col] = {'tstat': None, 'pvalue': None, 'n_FL5': int(len(a)), 'n_IgG': int(len(b))}
            except Exception:
                ttest_results = {'error': 'scipy not available; install scipy to run t-tests'}

    # finalize outputs
    merged_out = merged_df.copy()
    summary_out = summary.copy()

    return merged_out, summary_out, {'report': report, 'group_stats': group_stats, 'ttest': ttest_results}


# Example usage (not executed here):
# merged, summary, extras = correlate_with_cleaned_data(
#     cleaned_dir=r"c:\Users\hanss\git\DSCwashumed\Dexa Data Cleaning\cleaned_output",
#     bone_df=bone_merged_df,
#     join_on='subject_id',
#     baseline_labels=['Baseline','Week_0'],
#     followup_label='Week_4'
# )
# display(summary.head())
